In [ ]:

# Домашнє завдання: Логістична регресія — Rain in Australia
# Крок 1: Імпорт пакетів

import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import classification_report




In [ ]:

# Крок 2: Завантаження даних

url = "https://raw.githubusercontent.com/dsgt-kaggle-clones/rainforest-connection-species-audio-detection/master/data/weatherAUS.csv"
# Якщо URL не працює — завантажте локально:
# df = pd.read_csv('weatherAUS.csv')
try:
    df = pd.read_csv(url)
except:
    # Спробуємо інший шлях
    df = pd.read_csv('weatherAUS.csv')

print("Розмір датасету:", df.shape)
print(df.head())




In [ ]:

# Крок 3: Коректне розділення даних

# 3.1. Видалення ознак з великою кількістю пропущених значень (>40%)
threshold = 0.4
missing_ratio = df.isnull().mean()
cols_to_drop = missing_ratio[missing_ratio > threshold].index.tolist()
print("Видаляємо колонки:", cols_to_drop)
df = df.drop(columns=cols_to_drop)

# Також видаляємо рядки де немає цільової змінної
df = df.dropna(subset=['RainTomorrow'])

# Кодуємо цільову змінну
df['RainTomorrow'] = df['RainTomorrow'].map({'Yes': 1, 'No': 0})

# 3.2. Підмножини числових та категоріальних ознак
# (без Date, RainTomorrow)
exclude_cols = ['Date', 'RainTomorrow']
num_cols = df.select_dtypes(include=['float64', 'int64']).columns.tolist()
cat_cols = df.select_dtypes(include=['object']).columns.tolist()

# Прибираємо зайві
num_cols = [c for c in num_cols if c not in exclude_cols]
cat_cols = [c for c in cat_cols if c not in exclude_cols]

print("Числові ознаки:", num_cols)
print("Категоріальні ознаки:", cat_cols)

# 3.3. Перетворення Date → datetime, створення Year та Month
df['Date'] = pd.to_datetime(df['Date'])
df['Year'] = df['Date'].dt.year
df['Month'] = df['Date'].dt.month

# 3.4. Year → числова, Month → категоріальна
num_cols.append('Year')
cat_cols.append('Month')

# Перетворюємо Month на str для OHE
df['Month'] = df['Month'].astype(str)

# 3.5. Розділення за останнім роком
max_year = df['Year'].max()
print(f"Максимальний рік (тест): {max_year}")

train_mask = df['Year'] < max_year
test_mask = df['Year'] == max_year

X_train_num = df.loc[train_mask, num_cols]
X_test_num = df.loc[test_mask, num_cols]

X_train_cat = df.loc[train_mask, cat_cols]
X_test_cat = df.loc[test_mask, cat_cols]

y_train = df.loc[train_mask, 'RainTomorrow']
y_test = df.loc[test_mask, 'RainTomorrow']

print(f"Train size: {X_train_num.shape[0]}, Test size: {X_test_num.shape[0]}")




In [ ]:

# Крок 4: Відновлення пропущених значень (SimpleImputer)

# Числові — медіана
num_imputer = SimpleImputer(strategy='median')
X_train_num_imp = num_imputer.fit_transform(X_train_num)
X_test_num_imp = num_imputer.transform(X_test_num)

# Категоріальні — найчастіше значення
cat_imputer = SimpleImputer(strategy='most_frequent')
X_train_cat_imp = cat_imputer.fit_transform(X_train_cat)
X_test_cat_imp = cat_imputer.transform(X_test_cat)




In [ ]:

# Крок 5: Нормалізація числових ознак (StandardScaler)

scaler = StandardScaler()
X_train_num_scaled = scaler.fit_transform(X_train_num_imp)
X_test_num_scaled = scaler.transform(X_test_num_imp)




In [ ]:

# Крок 6: Кодування категоріальних ознак (OneHotEncoder)

ohe = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
X_train_cat_enc = ohe.fit_transform(X_train_cat_imp)
X_test_cat_enc = ohe.transform(X_test_cat_imp)

# Об'єднання числових та категоріальних ознак
X_train = np.hstack([X_train_num_scaled, X_train_cat_enc])
X_test = np.hstack([X_test_num_scaled, X_test_cat_enc])

print("Фінальний розмір X_train:", X_train.shape)
print("Фінальний розмір X_test:", X_test.shape)




In [ ]:

# Крок 7: Побудова моделі LogisticRegression

# Експеримент з різними solver
for solver in ['lbfgs', 'liblinear', 'saga']:
    model = LogisticRegression(solver=solver, max_iter=1000, random_state=42)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    print(f"\n=== Solver: {solver} ===")
    print(classification_report(y_test, y_pred, target_names=['No Rain', 'Rain']))




In [ ]:

# Крок 8: Фінальна модель та висновки

model_final = LogisticRegression(solver='lbfgs', max_iter=1000, random_state=42)
model_final.fit(X_train, y_train)
y_pred_final = model_final.predict(X_test)

print("\n=== ФІНАЛЬНІ МЕТРИКИ (lbfgs) ===")
print(classification_report(y_test, y_pred_final, target_names=['No Rain', 'Rain']))




ВИСНОВКИ:

1. Коректне розділення даних за часовим принципом (тест = останній рік)
   дозволяє об'єктивно оцінити здатність моделі до прогнозування
   на даних, які вона не бачила під час навчання.

2. Порівняно з базовою моделлю (випадкове розділення):
   - Метрики можуть бути дещо нижчими, оскільки умови тестування
     наближені до реальних (модель не "підглядала" в майбутнє)
   - Це більш чесна оцінка якості моделі

3. Логістична регресія показує:
   - Хорошу точність для класу "No Rain" (більший клас)
   - Нижчу точність для класу "Rain" через дисбаланс класів
   - Різні solver дають схожі результати при достатній кількості ітерацій

4. Для покращення моделі можна розглянути:
   - Балансування класів (class_weight='balanced')
   - Додаткові ознаки або feature engineering
   - Інші алгоритми (Random Forest, Gradient Boosting)
"""
print("Висновки записані у коді вище.")